In [ ]:
from scipy import stats
import pandas as pd
import numpy as np


### Предобработка исходных биржевых данных

1. Добавляем таргет
1. Удаляем пропуски, добавляем признаки аномалий цен и объема
2. Добавляем признаки, полученные из исходных данных, путем сдвигов с разными лагами, логарифмировании и пр
3. Добавляем технические индикаторы MACD и TEMA 
4. Добавляем как признак наличие дивидентного гэпа за определенную дату
5. Нормализуем данные с помощью StandardScaler

In [ ]:

def find_anomalies(series, threshold=3):
    z_scores = stats.zscore(series)
    return (z_scores > threshold) | (z_scores < -threshold)


In [ ]:
# 
def add_target(df):
    df['close_next_hour'] = df['Close'].shift(-30)
    df['target'] = (df['close_next_hour'] > df['Close']).astype(int)
    df.dropna(subset=['close_next_hour', 'target'])

    return df


In [ ]:
def clean_data(df):
	new_columns = []
	df.dropna(subset=['Close', 'Volume'], inplace=True)

	df['anomalies_price'] = find_anomalies(df['Close'])
	df['anomalies_volume'] = find_anomalies(df['Volume'])

	new_columns.append('anomalies_price')
	new_columns.append('anomalies_volume')

	return df, new_columns

In [ ]:

def create_features(df):
	features = ['Open', 'High', 'Low', 'Close', 'Volume']
	lag_periods= [3, 5, 7]
	new_columns = []

	for feature in features:
		df[f'{feature}_ratio_1'] = df[feature] / df[feature].shift(1)
		new_columns.append(f'{feature}_ratio_1')

		df[f'{feature}_log_diff_1'] = np.log(df[feature] / df[feature].shift(1))
		new_columns.append(f'{feature}_log_diff_1')

		for lag_period in lag_periods:
			# Momentum (разница между текущим значением и значением N периодов назад)
			df[f'{feature}_momentum_{lag_period}'] = df[feature] - df[feature].shift(lag_period)
			new_columns.append(f'{feature}_momentum_{lag_period}')

			# Rate of Change (ROC): процентное изменение за N периодов
			df[f'{feature}_roc_{lag_period}'] = (df[feature] - df[feature].shift(lag_period)) / df[feature].shift(lag_period) * 100
			new_columns.append(f'{feature}_roc_{lag_period}')

			# Exponential Moving Average (EMA) с периодом N
			df[f'{feature}_ema_{lag_period}'] = df[feature].ewm(span=lag_period, adjust=False).mean()
			new_columns.append(f'{feature}_ema_{lag_period}')

    # Удаление строк с NaN значениями, которые появились из-за сдвигов
	df = df.dropna()

	return df, new_columns

In [ ]:
def calculate_macd(df, feature, short_window=12, long_window=26):
    df = df.copy()

    # Рассчитываем краткосрочное и долгосрочное EMA
    ema_short = df[feature].ewm(span=short_window, adjust=False).mean()
    ema_long = df[feature].ewm(span=long_window, adjust=False).mean()

    # Разница между краткосрочным и долгосрочным EMA (MACD)
    df[f'{feature}_macd'] = ema_short - ema_long

    return df, f'{feature}_macd'

In [ ]:
def calculate_tema(df, feature, span):
    ema1 = df[feature].ewm(span=span, adjust=False).mean()
    ema2 = ema1.ewm(span=span, adjust=False).mean()
    ema3 = ema2.ewm(span=span, adjust=False).mean()

    df[f'{feature}_tema']  = 3 * ema1 - 3 * ema2 + ema3
    return df, f'{feature}_tema'

In [ ]:
def add_dividends(df, dividend_dates):
    dividend_dates = pd.to_datetime(dividend_dates, format='%d.%m.%Y %H:%M:%S')
    
    df['dividend'] = df['Datetime'].isin(dividend_dates)
    
    return df, 'dividend'

In [ ]:
def prepare_data(df, dividend_dates):
	new_columns = []

	df = add_target(df)

	df, new_columns_clean = clean_data(df)
	new_columns.extend(new_columns_clean)

	df, new_columns_features = create_features(df)
	new_columns.extend(new_columns_features)

	df, macd_column = calculate_macd(df, 'Close')
	new_columns.append(macd_column)

	df, tema_column = calculate_tema(df, 'Close', span=3)
	new_columns.append(tema_column)

	# df, dividend_column = add_dividends(df, dividend_dates)
	# new_columns.append(dividend_column)

	df = df[['target', 'Datetime']+new_columns]

	return df, new_columns
    

In [ ]:
# Даты дивидендных гэпов Сбера
dividend_dates = [
    '11.07.2024 10:00:00', 
    '11.05.2023 10:00:00', 
    '12.05.2021 10:00:00'
]

In [ ]:
# Тестирование

# df_test = pd.read_csv('sber_data.csv')
# df_test['time'] = pd.to_datetime(df_test['time'])
# df_processed, new_columns = prepare_data(df_test, dividend_dates)
# len(new_columns)

60

In [ ]:
#df_processed

,target,time,anomalies_price,anomalies_volume,open_ratio_1,open_log_diff_1,open_momentum_3,open_roc_3,open_ema_3,open_momentum_5,...,volume_ema_3,volume_momentum_5,volume_roc_5,volume_ema_5,volume_momentum_7,volume_roc_7,volume_ema_7,close_macd,close_tema,dividend
7,0,2023-04-04 14:30:00+00:00,False,False,1.000000,0.000000,0.0,0.000000,215.000000,0.0,...,5556.109375,-8976.0,-77.640343,6295.498400,1518.0,142.268041,6241.065796,0.000000,215.000000,False
8,0,2023-04-04 14:31:00+00:00,False,False,1.000000,0.000000,0.0,0.000000,215.000000,0.0,...,4920.554688,-229.0,-5.073106,5625.332266,-1724.0,-28.690298,5752.049347,0.000000,215.000000,False
9,0,2023-04-04 14:32:00+00:00,False,False,1.000000,0.000000,0.0,0.000000,215.000000,0.0,...,3380.277344,-9018.0,-83.053969,4363.554844,-9721.0,-84.084422,4774.037010,0.000000,215.000000,False
10,0,2023-04-04 14:33:00+00:00,False,False,1.000000,0.000000,0.0,0.000000,215.000000,0.0,...,2498.138672,-10990.0,-87.180708,3447.703230,-2898.0,-64.200266,3984.527758,0.000000,215.000000,False
11,0,2023-04-04 14:34:00+00:00,False,True,1.000000,0.000000,0.0,0.000000,215.000000,0.0,...,19071.569336,29116.0,445.948844,14180.135486,24787.0,228.283293,11899.645818,0.000000,215.000000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411444,0,2024-09-25 14:14:00+00:00,False,False,1.000000,0.000000,-1.0,-0.370370,269.250244,-1.0,...,22871.214432,12156.0,71.937507,23441.076905,19295.0,197.714930,23913.564664,-0.347806,269.813741,False
411445,0,2024-09-25 14:15:00+00:00,False,False,1.003717,0.003711,0.0,0.000000,269.625122,0.0,...,16897.107216,-6015.0,-35.511867,19268.384604,-135406.0,-92.535314,20665.923498,-0.311150,270.032017,False
411446,0,2024-09-25 14:16:00+00:00,False,False,1.000000,0.000000,1.0,0.371747,269.812561,0.0,...,24387.053608,14531.0,83.771475,23471.256402,14979.0,88.643626,23468.692624,-0.358658,269.180151,False
411447,0,2024-09-25 14:17:00+00:00,False,False,0.996296,-0.003711,0.0,0.000000,269.406281,-1.0,...,16078.526804,-10490.0,-57.447974,18237.504268,-9168.0,-54.126815,19544.019468,-0.391792,268.972932,False
